# Multi-Task Sentiment Analysis: Ablation Study
This notebook analyzes the impact of individual task heads on overall performance by comparing models trained with full multitask objectives versus those with ablated architectures.

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Styling
plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'monospace', 'axes.edgecolor': '#2A2A2A', 'axes.facecolor': '#111111', 'figure.facecolor': '#0A0A0A'})

def load_history(name):
    path = f'../checkpoints/{name}_history.json'
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return []

In [ ]:
# 1. Ablation Overview Table
experiments = ['full_model', 'no_sarcasm', 'no_emotion', 'frozen_encoder']
metrics = []
for exp in experiments:
    hist = load_history(exp)
    if hist:
        best_ep = max(hist, key=lambda x: x['overall_score'])
        metrics.append({
            'Experiment': exp,
            'Overall Score': best_ep['overall_score'],
            'Sentiment F1': best_ep['sentiment_f1_macro'],
            'Emotion F1': best_ep['emotion_f1_macro'],
            'Sarcasm F1': best_ep['sarcasm_f1_macro']
        })
if metrics:
    df = pd.DataFrame(metrics)
    display(df)
else:
    print('Simulated runs missing, please run train.py first.')

In [ ]:
# 2. Training Loss Curves
plt.figure(figsize=(10, 5))
colors = ['#E8FF47', '#4ADE80', '#F87171', '#94A3B8']
for i, exp in enumerate(experiments):
    hist = load_history(exp)
    if hist:
        losses = [ep['val_loss'] for ep in hist]
        plt.plot(range(1, len(losses)+1), losses, label=exp, color=colors[i], marker='o')
plt.title('Validation Loss across Ablations')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.15)
plt.legend()
plt.show()

In [ ]:
# 3. Sarcastic vs Non-Sarcastic Accuracy
if metrics:
    sarc_acc = []
    nonsarc_acc = []
    for exp in experiments:
        hist = load_history(exp)
        if hist:
            best_ep = max(hist, key=lambda x: x.get('overall_score', 0))
            sarc_acc.append(best_ep.get('sentiment_accuracy_sarcastic', 0))
            nonsarc_acc.append(best_ep.get('sentiment_accuracy_nonsarcastic', 0))
    
    x = np.arange(len(sarc_acc))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    rects1 = ax.bar(x - width/2, sarc_acc, width, label='Sarcastic', color='#FB923C')
    rects2 = ax.bar(x + width/2, nonsarc_acc, width, label='Non-Sarcastic', color='#E8FF47')
    ax.set_ylabel('Sentiment Accuracy')
    ax.set_title('Sentiment Performance by Sarcasm Presence')
    ax.set_xticks(x)
    ax.set_xticklabels([e.replace('_', '\n') for e in experiments if load_history(e)])
    ax.legend()
    plt.show()

In [ ]:
# 4. Reliability Diagram (ECE)
def plot_reliability_diagram(ece_score):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], '--', label='Perfect Calibration', color='#94A3B8')
    confs = np.linspace(0.1, 0.9, 10)
    accs = confs * 0.95 + np.random.normal(0, ece_score, 10)
    ax.plot(confs, accs, marker='s', color='#4ADE80', label=f'Full Model (mocked ECE={ece_score:.3f})')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_title('Reliability Diagram')
    ax.legend()
    plt.grid(alpha=0.1)
    plt.show()

hist = load_history('full_model')
if hist:
    plot_reliability_diagram(hist[-1]['ece'])

In [ ]:
# 5. Attention Visualization Hook
from IPython.display import display, HTML

print('To render heatmaps inter-temporally:')
print('-----------------------------------------')
print('from explain.attention_viz import get_attention_heatmap')
print('out = get_attention_heatmap(model, tokenizer, "This is a great test, but maybe not.")')
print('plt.imshow(out["raw_attention_matrix"][0, :10, :10], cmap="viridis")')
